# Spline Polynomials
In the code below, we synthesize a spline of a specified degree, delay, period, with random spline coefficients. We display this spline in thick gray, with blue stems and rings that highlight the samples at the integers, and red stems and rings that highlight the samples at the boundaries of one period. The black dots give the knots of the spline—the places where the polynomial pieces meet. We then extract from this random spline the list of its polynomial pieces, which we evaluate and overlay as lines of alternating green and brown colors. The spline curve and the polynomial curves match perfectly. Finally, we print a verbose description of the polynomial pieces.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Persistent spline
f = sk.PeriodicSpline1D()
f.spline_coeff[0] = rng.standard_normal()

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0
):
    global f # Spline

    # Update of the period
    c = f.spline_coeff
    if f.period < period:
        c = np.append(c, rng.standard_normal(period - f.period))
    else:
        c = c[ : period]
    f = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree, delay = delay)
    # Plot the spline
    f.plot(
        plt.subplots(),
        plotpoints = 301,
        curve_fmt = "#E0E0E0", # Light gray
        curve_lw = 9 # Thick
    )

    # Polynomial pieces
    pieces = f.piecewise_polynomials()
    # Plot each piece independently, with alternating colors
    fmt = ["-C2", "-C5"]
    for piece in pieces.pieces:
        lb = piece.domain.infimum # Lower bound of the domain
        ub = piece.domain.supremum # Upper bound of the domain
        if lb == ub: # Ignore degenerate domains
            continue
        abscissa = np.linspace(lb, ub, num = 15, endpoint = True)
        polynomial = piece.item
        data = np.array([float(polynomial(x)) for x in abscissa], dtype = "float")
        plt.plot(abscissa, data, fmt[0])
        fmt = [fmt[1], fmt[0]]

    # Show the plot
    plt.show()

    # Verbose description
    print("---")
    print("Degree =", degree)
    print("Delay =", delay)
    print("Period =", period)
    print("Spline coefficients =", c)
    print("---")
    for piece in pieces.pieces:
        print(piece)
        print()

widgets.interactive(
    update_plot,
    period = (1, max_period),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay)
)